In [ ]:
import numpy as np

def norm_w(w, alpha=1):
    # alpha = 0: keep raw weights
    # alpha = 1: full min-max normalization

    w_abs = np.abs(w)
    min_vals = np.min(w_abs)
    max_vals = np.max(w_abs)
    w_minmax = ((w_abs - min_vals) / (max_vals - min_vals + 1e-6)) + 1e-6

    # Soft mixture
    return (1 - alpha) * w_abs + alpha * w_minmax

In [ ]:


def w_norm(w, mean = 0.1, std = 0.1):
    w_abs = np.abs(w)
    w_mean = np.mean(w)
    w_std = np.std(w)
    w_norm = np.abs((w-w_mean)/(10*w_std))
    # w_norm = norm_w(w_norm)
    print(w_std, w_mean)
    return w_norm

In [ ]:
def w_norm_exp(w, mean = 0.1, std = 0.1):
    w_abs = np.abs(w)
    w_mean = np.mean(w)
    w_std = np.std(w)
    w_norm = np.abs((w)/(10*w_std))
    w_norm = np.exp(w_norm)
    print(w_mean, w_std)
    return w_norm

In [ ]:
from scipy.stats import skew, kurtosis
from scipy.stats import probplot

def is_gaussian(data, skew_thresh=1.0, kurt_thresh=1.0):
    osm, osr = probplot(data, dist="norm", fit=False)
    corr = np.corrcoef(osm, osr)[0,1]
    print("QQ-correlation:", corr)

    return corr > 0.99



In [ ]:


def w_norm_gauss(w, mean = 0.1, std = 0.1):
    flag=is_gaussian(w)
    w_mean = np.mean(w)
    w_std = np.std(w)
    w_norm = np.abs((w-w_mean)/(10*w_std))
    
    w_abs = np.abs(w)
    
    if flag:
        print("Likely Gaussian")
        w_norm = norm_w(w_abs)
    else: 
        w_norm = w_norm
        print("Not Gaussian")

    return w_norm

In [ ]:
# pip install torch matplotlib  # if needed

import torch
import matplotlib.pyplot as plt
import numpy as np

# ckpt_path = "CNN/models/vgg9_ori_10_tanh_s2.pth"   # <-- change this
ckpt_path = "CNN/models/new/cnn_ori_relu.pth"

# --- 1) Load checkpoint on CPU ---
obj = torch.load(ckpt_path, map_location="cpu")

if isinstance(obj, dict) and any(isinstance(v, torch.Tensor) for v in obj.values()):
    state_dict = obj
elif isinstance(obj, dict) and "state_dict" in obj and isinstance(obj["state_dict"], dict):
    state_dict = obj["state_dict"]
elif hasattr(obj, "state_dict"):
    state_dict = obj.state_dict()
else:
    raise ValueError("Couldn't find a state_dict in this checkpoint.")

# --- 2) Keep only floating-point ".weight" tensors (no biases/BN stats) ---
param_items = []
for name, tensor in state_dict.items():
    if (
        isinstance(tensor, torch.Tensor)
        and tensor.dtype.is_floating_point
        and name.endswith(".weight")
    ):
        param_items.append((name, tensor))

if len(param_items) == 0:
    raise ValueError("No floating-point .weight tensors found.")

# --- 3) Derive layer names and pick the LAST THREE layers ---
# Define a "layer" as everything before the final ".weight" (e.g., "encoder.layers.23.mlp.fc2")
ordered_layers = []
seen = set()
for name, _ in param_items:
    layer = name.rsplit(".", 1)[0]
    if layer not in seen:
        ordered_layers.append(layer)
        seen.add(layer)
        
print(ordered_layers)

last_three_layers = ordered_layers
print("Selected layers (last 3):")
for L in last_three_layers:
    print("  ", L)

# --- 4) Collect weights for those three layers ---
selected = []
for name, tensor in param_items:
    layer = name.rsplit(".", 1)[0]
    if layer in last_three_layers:
        selected.append((layer, tensor.detach().cpu().reshape(-1)))

if not selected:
    raise ValueError("No matching .weight tensors for the last three layers.")

# --- 5) Plot a combined histogram of the last-3 layers' weights ---
combined = torch.cat([t for _, t in selected]).numpy()
plt.figure()
plt.hist((combined), bins=100)
plt.title("Histogram of Last 3 Layers' Weights (combined)")
plt.xlabel("Weight value")
plt.ylabel("Count")
plt.yscale("linear")   # use "log" if very peaky
plt.show()

# --- 6) (Optional) Per-layer histograms ---
for layer in last_three_layers:
    layer_weights = torch.cat([t for L, t in selected if L == layer]).numpy()
    plt.figure()
    # layer_weights_norm = w_norm_gauss(layer_weights)
    layer_weights_norm1 = norm_w(layer_weights)
    print(np.min(np.abs(layer_weights)))
    # plt.hist((layer_weights_norm1), bins=100)
    # plt.hist(np.abs(layer_weights_norm), bins=100)
    
    plt.hist(np.abs(layer_weights), bins=100)
    
    # l_std = np.std(layer_weights_norm)
    # l_mean = np.mean(layer_weights_norm)
    # print(l_std, l_mean)
    
    # l_std1 = np.std(layer_weights_norm1)
    # l_mean1 = np.mean(layer_weights_norm1)
    # print(l_std1, l_mean1)
    
    plt.title(f"Histogram: {layer}")
    plt.xlabel("Weight value")
    plt.ylabel("Count")
    plt.yscale("linear")  # or "log"
    plt.show()


In [ ]:
import torch
import torch.nn.functional as F

q = torch.randint(1,5,(2,2,2))
k = torch.randint(1,5,(2,2,2))
v = torch.randint(1,5,(2,2,2))
print("q = ", q)
print()
print("k = ", k)
print()

print(k.transpose(-2,-1))
print()
print(q@k)
print()
print(q @ k.transpose(-2,-1))

In [1]:
def _reshape_for_heads(x, num_heads, head_dim):
    batch, seq_len, _ = x.shape
    return x.view(batch, seq_len, num_heads, head_dim).transpose(1, 2).contiguous()

In [2]:
def _repeat_kv(hidden_states, n_rep):
    if n_rep == 1:
        return hidden_states

    batch, num_kv_heads, seq_len, head_dim = hidden_states.shape
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_kv_heads, n_rep, seq_len, head_dim)
    return hidden_states.reshape(batch, num_kv_heads * n_rep, seq_len, head_dim)

In [11]:
import torch
import torch.nn.functional as F

# q = torch.rand(1,2,8)
# k = torch.rand(1,2,4)
# v = torch.rand(1,2,4)

a = torch.arange(16)
q = a.reshape(1,2,8)

b = torch.arange(8)
k = b.reshape(1,2,4)

q_1 = _reshape_for_heads(q, 4, 2)
k_1 = _reshape_for_heads(k, 2, 2)

print(q)
print()
print(q_1)

print()
print(k)
print()
print(k_1)

k_2 = _repeat_kv(k_1, 2)
print()
print(k_2.transpose(-2, -1))

print()
print(q_1 @ k_2.transpose(-2, -1))

tensor([[[ 0,  1,  2,  3,  4,  5,  6,  7],
         [ 8,  9, 10, 11, 12, 13, 14, 15]]])

tensor([[[[ 0,  1],
          [ 8,  9]],

         [[ 2,  3],
          [10, 11]],

         [[ 4,  5],
          [12, 13]],

         [[ 6,  7],
          [14, 15]]]])

tensor([[[0, 1, 2, 3],
         [4, 5, 6, 7]]])

tensor([[[[0, 1],
          [4, 5]],

         [[2, 3],
          [6, 7]]]])

tensor([[[[0, 4],
          [1, 5]],

         [[0, 4],
          [1, 5]],

         [[2, 6],
          [3, 7]],

         [[2, 6],
          [3, 7]]]])

tensor([[[[  1,   5],
          [  9,  77]],

         [[  3,  23],
          [ 11,  95]],

         [[ 23,  59],
          [ 63, 163]],

         [[ 33,  85],
          [ 73, 189]]]])
